In [ ]:
from huggingface_hub import login

# Paste your token here
login("------------")

/Users/mlf25/Documents/Entalpic/LLM_synth/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset

# Load the "full" configuration and split
dataset = load_dataset(
    "LeMaterial/LeMat-Synth-Papers",
    "full",          # configuration/subset
    split=None,      # other splits include 'arxiv', 'omg24', 'chemrxiv'
    token=True    
)

#Print column names
print(dataset.column_names)


{'arxiv': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis'], 'omg24': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis'], 'chemrxiv': ['id', 'title', 'authors', 'abstract', 'doi', 'published_date', 'updated_date', 'categories', 'license', 'pdf_url', 'views_count', 'read_count', 'citation_count', 'keywords', 'text_paper', 'text_si', 'source', 'pdf_extractor', 'images', 'structured_synthesis']}


In [3]:
#Query function, takes in the split name, which column to check and a list of keywords to return the IDs of the papers that match
def query_db(text_column,split_name,list_keywords):
    ds = dataset[split_name]

    results = {}  # store keyword and corresponding IDs

    for keyword in list_keywords:

        # define filter function for this keyword
        def contains_keyword(example, kw=keyword):
            return (
                example[text_column] is not None 
                and kw.lower() in example[text_column].lower()
            )

        # filter dataset
        filtered = ds.filter(contains_keyword)

        # extract IDs
        ids = filtered["id"]

        # store in dictionary
        results[keyword] = ids

    # Print results
    for kw, ids in results.items():
        print(f"\nKeyword: {kw}")
        print(f"Found {len(ids)} entries")
        print(ids)
    
    return results

In [4]:
#From a given subset of papers, removes redundancies 
def return_nonredundant_ids(results):
    all_ids = []

    for kw, ids in results.items():
        all_ids.extend(ids)

    # Convert to a set to remove duplicates
    unique_ids = set(all_ids)

    # Count them
    print("Number of papers containing at least one keyword:", len(unique_ids))

    return(unique_ids)

In [7]:
import pickle

split_name_list = ["arxiv", "omg24", "chemrxiv"] 
text_column = "abstract"
list_keywords = ['heterogeneous catalysis','heterogeneous catalyst','was heated at','was heated under', 'thermal treatment', 'was measured at different temperatures', 'activation energy','variations','conversion', 'efficiency','activation energy']
'''list_keywords = ["catalyst", "catalysis", "catalytic", "TOF", "activation energy"]'''
db = {}

for split_name in split_name_list:
    result = query_db(split_name=split_name, text_column=text_column, list_keywords=list_keywords)
    nonredundant_ids = return_nonredundant_ids(result)
    print(split_name, len(nonredundant_ids), nonredundant_ids)
    
    db[split_name] = nonredundant_ids


#export db to pickle file
with open("db_thermocatalysis.pkl", "wb") as f:
    pickle.dump(db, f)

print("Saved db.pkl with keys:", list(db.keys()))



Keyword: heterogeneous catalysis
Found 11 entries
Column(['1904.05666', '1905.06271', '2102.08269', '2112.05953', '2201.05853'])

Keyword: heterogeneous catalyst
Found 12 entries
Column(['1309.4319', '1706.01493', '1709.02248', '1903.02363', '2003.01594'])

Keyword: was heated at
Found 2 entries
Column(['1901.06784', '2411.10331'])

Keyword: was heated under
Found 0 entries
Column([])

Keyword: thermal treatment
Found 121 entries
Column(['0704.2761', '0710.2392', '0712.4197', '0901.1591', '0902.4104'])

Keyword: was measured at different temperatures
Found 0 entries
Column([])

Keyword: activation energy
Found 419 entries
Column(['0704.0538', '0708.1135', '0708.1610', '0709.1343', '0710.1847'])

Keyword: variations
Found 902 entries
Column(['0705.2812', '0705.4375', '0706.0821', '0707.0225', '0707.0522'])

Keyword: conversion
Found 1179 entries
Column(['0707.1434', '0708.4050', '0710.5585', '0712.2139', '0712.2982'])

Keyword: efficiency
Found 1962 entries
Column(['0707.1434', '0707.1